# 04 · Subqueries and CTEs (Spark)

A **subquery** is a query inside another query.
- **scalar** subquery: returns one value, e.g. `(SELECT AVG(salary) FROM employees)`
- **correlated** subquery: uses a column from the outer query, so it runs "per row"
- `IN` / `EXISTS` / `NOT EXISTS`: check if matching rows exist

A **CTE** (`WITH name AS (...)`) gives a subquery a name. It makes long queries easy to read, step by step.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/04_subqueries_ctes.ipynb`](../../spark/practice/04_subqueries_ctes.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. Above the company average  ·  🟢 easy

Show `name` and `salary` of employees who earn more than the **average salary of all employees**.

*Expected result: 10 rows · columns: `name`, `salary`*

**How it works:** The subquery `(SELECT AVG(salary) FROM employees)` returns one number; the outer WHERE compares every salary with it.

In [ ]:
%%sql
SELECT name, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)

,name,salary
0,Alice Johnson,250000.00
1,Bob Smith,150000.00
2,Carol White,120000.00
3,David Brown,120000.00
4,Eva Green,110000.00
5,Henry Clark,140000.00
6,Nathan King,130000.00
7,Rachel Nelson,135000.00
8,Victor Roberts,105000.00
9,Yara Campbell,115000.00


**DataFrame API:**

In [ ]:
avg_sal = employees.agg(F.avg("salary")).first()[0]
res = employees.filter(F.col("salary") > avg_sal).select("name", "salary")
res.show()

+--------------+---------+
|          name|   salary|
+--------------+---------+
| Alice Johnson|250000.00|
|     Bob Smith|150000.00|
|   Carol White|120000.00|
|   David Brown|120000.00|
|     Eva Green|110000.00|
|   Henry Clark|140000.00|
|   Nathan King|130000.00|
| Rachel Nelson|135000.00|
|Victor Roberts|105000.00|
| Yara Campbell|115000.00|
+--------------+---------+

**Why / traps:** `.first()[0]` brings one value to the driver (your Python program). Fine for one number; never do it for big data.

## Q2. Above their department average (correlated)  ·  🟡 medium

Show `name`, `dept_id`, `salary` of employees who earn more than the average salary **of their own department**.

*Expected result: 7 rows · columns: `name`, `dept_id`, `salary`*

**How it works:** For each employee row, the subquery averages salaries of **the same** department (`x.dept_id = e.dept_id`), and WHERE keeps people above it.

In [ ]:
%%sql
SELECT e.name, e.dept_id, e.salary
FROM employees e
WHERE e.salary > (SELECT AVG(x.salary)
                  FROM employees x
                  WHERE x.dept_id = e.dept_id)

,name,dept_id,salary
0,Alice Johnson,1,250000.00
1,Bob Smith,1,150000.00
2,Henry Clark,2,140000.00
3,Nathan King,3,130000.00
4,Rachel Nelson,4,135000.00
5,Victor Roberts,5,105000.00
6,Yara Campbell,6,115000.00


**DataFrame API:**

In [ ]:
dept_avg = employees.groupBy("dept_id").agg(F.avg("salary").alias("dept_avg"))
res = (employees.join(dept_avg, "dept_id")
       .filter(F.col("salary") > F.col("dept_avg"))
       .select("name", "dept_id", "salary"))
res.show()

+--------------+-------+---------+
|          name|dept_id|   salary|
+--------------+-------+---------+
| Alice Johnson|      1|250000.00|
|     Bob Smith|      1|150000.00|
|   Henry Clark|      2|140000.00|
|   Nathan King|      3|130000.00|
| Rachel Nelson|      4|135000.00|
|Victor Roberts|      5|105000.00|
| Yara Campbell|      6|115000.00|
+--------------+-------+---------+

**Why / traps:** The inner query uses `e.dept_id` from the outer query: that makes it **correlated**. The engine usually rewrites it as a join with a GROUP BY (like the DataFrame answer). The employee with NULL dept_id drops out: `x.dept_id = NULL` never matches.

## Q3. Not a manager: the NOT IN trap  ·  🟡 medium

Show `emp_id` and `name` of employees who are **nobody's manager**.

First try `WHERE emp_id NOT IN (SELECT manager_id FROM employees)`. How many rows? Why?

*Expected result: 18 rows · columns: `emp_id`, `name`*

**How it works:** `NOT EXISTS` asks, for each employee: is there anyone whose `manager_id` is my `emp_id`? If not, I manage nobody. No NULL problem.

In [ ]:
%%sql
SELECT e.emp_id, e.name
FROM employees e
WHERE NOT EXISTS (SELECT 1 FROM employees x WHERE x.manager_id = e.emp_id)

,emp_id,name
0,4,David Brown
1,5,Eva Green
2,6,Frank Moore
3,7,Grace Lee
4,10,Jack Walker
5,11,Karen Hall
6,12,Leo Allen
7,13,Mia Young
8,16,Paul Adams
9,17,Quinn Baker


**DataFrame API:**

In [ ]:
mgrs = employees.select(F.col("manager_id").alias("emp_id")).where("emp_id IS NOT NULL")
res = employees.join(mgrs, "emp_id", "left_anti").select("emp_id", "name")
res.show()

+------+---------------+
|emp_id|           name|
+------+---------------+
|     4|    David Brown|
|     5|      Eva Green|
|     6|    Frank Moore|
|     7|      Grace Lee|
|    10|    Jack Walker|
|    11|     Karen Hall|
|    12|      Leo Allen|
|    13|      Mia Young|
|    16|     Paul Adams|
|    17|    Quinn Baker|
|    20|  Tina Mitchell|
|    21|      Uma Perez|
|    23|   Wendy Turner|
|    24|Xavier Phillips|
|    27|      Amy Evans|
|    28|  Brian Edwards|
|    29|  Chloe Collins|
|    30| Daniel Stewart|
+------+---------------+

**Why / traps:** `NOT IN` with a NULL in the list returns **0 rows**: `5 NOT IN (1, NULL)` means `5 <> 1 AND 5 <> NULL`, which is unknown. Use `NOT EXISTS`, or add `WHERE manager_id IS NOT NULL` inside the subquery.

## Q4. Products in big orders (IN)  ·  🟢 easy

Show `product_id` and `product_name` of products that appear in at least one **completed** order with amount **over 200**. Each product once.

*Expected result: 6 rows · columns: `product_id`, `product_name`*

**How it works:** The subquery lists product ids from completed orders over 200; `IN` keeps products in that list. Duplicates in the list don't matter.

In [ ]:
%%sql
SELECT product_id, product_name
FROM products
WHERE product_id IN (
    SELECT i.product_id
    FROM order_items i
    JOIN orders o ON o.order_id = i.order_id
    WHERE o.status = 'completed' AND o.amount > 200)

,product_id,product_name
0,2,USB-C Cable
1,3,Bluetooth Speaker
2,6,Headphones
3,8,Data Engineering 101
4,13,Water Bottle
5,15,T-Shirt


**DataFrame API:**

In [ ]:
big = (order_items.join(orders, "order_id")
       .filter((F.col("status") == "completed") & (F.col("amount") > 200)))
res = products.join(big, "product_id", "left_semi").select("product_id", "product_name")
res.show()

+----------+--------------------+
|product_id|        product_name|
+----------+--------------------+
|         2|         USB-C Cable|
|         3|   Bluetooth Speaker|
|         6|          Headphones|
|         8|Data Engineering 101|
|        13|        Water Bottle|
|        15|             T-Shirt|
+----------+--------------------+

## Q5. Departments that pay above average  ·  🟡 medium

Show `dept_id` and `avg_salary` (2 decimals) of departments whose average salary is higher than the company-wide average salary.

*Expected result: 2 rows · columns: `dept_id`, `avg_salary`*

**How it works:** Group employees by department and compute the average; `HAVING` compares each group's average with the company average from a subquery.

In [ ]:
%%sql
SELECT dept_id, ROUND(AVG(salary), 2) AS avg_salary
FROM employees
WHERE dept_id IS NOT NULL
GROUP BY dept_id
HAVING AVG(salary) > (SELECT AVG(salary) FROM employees)

,dept_id,avg_salary
0,4,107750.00
1,1,134285.71


**DataFrame API:**

In [ ]:
company = employees.agg(F.avg("salary")).first()[0]
res = (employees.filter(F.col("dept_id").isNotNull())
       .groupBy("dept_id").agg(F.avg("salary").alias("a"))
       .filter(F.col("a") > company)
       .select("dept_id", F.round("a", 2).alias("avg_salary")))
res.show()

+-------+----------+
|dept_id|avg_salary|
+-------+----------+
|      4| 107750.00|
|      1| 134285.71|
+-------+----------+

## Q6. Customers who spend above average (CTE)  ·  🟡 medium

Use a CTE. Step 1: total completed amount per customer (count each order once). Step 2: keep customers whose total is above the **average of those totals**. Show `customer_id`, `total_spent`. Highest first.

*Expected result: 11 rows · columns: `customer_id`, `total_spent`*

**How it works:** CTE 1 removes duplicates; CTE 2 sums spend per customer; the final query keeps customers above `AVG(total_spent)` of CTE 2.

In [ ]:
%%sql
WITH clean_orders AS (
    SELECT DISTINCT order_id, customer_id, amount, status
    FROM orders
),
spend AS (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM clean_orders
    WHERE status = 'completed'
    GROUP BY customer_id
)
SELECT customer_id, total_spent
FROM spend
WHERE total_spent > (SELECT AVG(total_spent) FROM spend)
ORDER BY total_spent DESC, customer_id

,customer_id,total_spent
0,28,740.00
1,29,535.00
2,27,492.00
3,26,450.00
4,30,390.00
5,31,385.00
6,25,330.00
7,14,312.00
8,16,306.00
9,20,290.00


**DataFrame API:**

In [ ]:
spend = (orders.dropDuplicates().filter(F.col("status") == "completed")
         .groupBy("customer_id").agg(F.sum("amount").alias("total_spent")))
avg_spend = spend.agg(F.avg("total_spent")).first()[0]
res = (spend.filter(F.col("total_spent") > avg_spend)
       .orderBy(F.desc("total_spent"), "customer_id"))
res.show()

+-----------+-----------+
|customer_id|total_spent|
+-----------+-----------+
|         28|     740.00|
|         29|     535.00|
|         27|     492.00|
|         26|     450.00|
|         30|     390.00|
|         31|     385.00|
|         25|     330.00|
|         14|     312.00|
|         16|     306.00|
|         20|     290.00|
|         22|     244.00|
+-----------+-----------+

**Why / traps:** A CTE can be used more than once (`spend` is used twice here). Each CTE is one clear step: easy to test alone.

## Q7. Second highest salary (no window functions)  ·  🟡 medium

Show the **second highest** different salary as `second_highest`. Do it without window functions.

*Expected result: 1 row · columns: `second_highest`*

**How it works:** Take the highest salary that is **lower** than the overall maximum = the second highest different salary.

In [ ]:
%%sql
SELECT MAX(salary) AS second_highest
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees)

,second_highest
0,150000.00


**DataFrame API:**

In [ ]:
top = employees.agg(F.max("salary")).first()[0]
res = employees.filter(F.col("salary") < top).agg(F.max("salary").alias("second_highest"))
res.show()

+--------------+
|second_highest|
+--------------+
|     150000.00|
+--------------+

**Why / traps:** Classic interview question. If there is no second salary, MAX returns NULL (one row with NULL), which is usually what they want. For the Nth salary see topic 05.

## Q8. Employee count per department (scalar subquery in SELECT)  ·  🟡 medium

Show every `dept_name` with `num_employees`, using a **subquery in the SELECT list**. Legal must show 0.

*Expected result: 7 rows · columns: `dept_name`, `num_employees`*

**How it works:** For each department, the subquery in SELECT counts its employees. It returns 0 for Legal (COUNT of no rows is 0).

In [ ]:
%%sql
SELECT d.dept_name,
       (SELECT COUNT(*) FROM employees e WHERE e.dept_id = d.dept_id) AS num_employees
FROM departments d
ORDER BY num_employees DESC, d.dept_name

,dept_name,num_employees
0,Engineering,7
1,Sales,6
2,Operations,5
3,Finance,4
4,Marketing,4
5,HR,3
6,Legal,0


**DataFrame API:**

In [ ]:
counts = employees.groupBy("dept_id").agg(F.count("*").alias("n"))
res = (departments.join(counts, "dept_id", "left")
       .select("dept_name", F.coalesce("n", F.lit(0)).alias("num_employees"))
       .orderBy(F.desc("num_employees"), "dept_name"))
res.show()

+-----------+-------------+
|  dept_name|num_employees|
+-----------+-------------+
|Engineering|            7|
|      Sales|            6|
| Operations|            5|
|    Finance|            4|
|  Marketing|            4|
|         HR|            3|
|      Legal|            0|
+-----------+-------------+

**Why / traps:** COUNT over zero rows gives 0 (not NULL). SUM over zero rows gives NULL.

## Q9. Departments with a recent hire (EXISTS)  ·  🟢 easy

Show `dept_name` of departments with **at least one** employee hired on or after 2023-01-01.

*Expected result: 4 rows · columns: `dept_name`*

**How it works:** `EXISTS` with a correlated subquery: keep a department if at least one of its employees has `hire_date >= 2023-01-01`.

In [ ]:
%%sql
SELECT d.dept_name
FROM departments d
WHERE EXISTS (SELECT 1
              FROM employees e
              WHERE e.dept_id = d.dept_id
                AND e.hire_date >= DATE '2023-01-01')

,dept_name
0,Sales
1,Marketing
2,HR
3,Operations


**DataFrame API:**

In [ ]:
recent = employees.filter(F.col("hire_date") >= F.lit("2023-01-01").cast("date"))
res = departments.join(recent, "dept_id", "left_semi").select("dept_name")
res.show()

+----------+
| dept_name|
+----------+
|     Sales|
| Marketing|
|        HR|
|Operations|
+----------+

**Why / traps:** `DATE '2023-01-01'` is a date literal (a fixed date written in SQL). It works in all 3 engines.

## Q10. The most recent order(s)  ·  🟢 easy

Show `order_id`, `customer_id`, `order_date` of the order(s) on the **latest order date**. If two orders share that date, show both. No duplicate rows.

*Expected result: 1 row · columns: `order_id`, `customer_id`, `order_date`*

**How it works:** The subquery finds the latest date; the outer query keeps all orders on that date. DISTINCT removes duplicate rows.

In [ ]:
%%sql
SELECT DISTINCT order_id, customer_id, order_date
FROM orders
WHERE order_date = (SELECT MAX(order_date) FROM orders)

,order_id,customer_id,order_date
0,1104,2,2025-12-28


**DataFrame API:**

In [ ]:
last_day = orders.agg(F.max("order_date")).first()[0]
res = orders.filter(F.col("order_date") == last_day).select("order_id", "customer_id", "order_date").distinct()
res.show()

+--------+-----------+----------+
|order_id|customer_id|order_date|
+--------+-----------+----------+
|    1104|          2|2025-12-28|
+--------+-----------+----------+

## Q11. Busy months of 2024 (CTE chain)  ·  🔴 hard

For year 2024: count different orders per month (`order_month` 1–12, `num_orders`). Keep only months with **more** orders than the average month of 2024.

*Expected result: 4 rows · columns: `order_month`, `num_orders`*

**How it works:** CTE `per_month` counts orders per month of 2024; the final query compares each month with the average of all months in the CTE.

In [ ]:
%%sql
WITH per_month AS (
    SELECT EXTRACT(MONTH FROM order_date) AS order_month,
           COUNT(DISTINCT order_id)        AS num_orders
    FROM orders
    WHERE order_date >= DATE '2024-01-01' AND order_date < DATE '2025-01-01'
    GROUP BY EXTRACT(MONTH FROM order_date)
)
SELECT order_month, num_orders
FROM per_month
WHERE num_orders > (SELECT AVG(num_orders) FROM per_month)
ORDER BY order_month

,order_month,num_orders
0,1,4
1,3,5
2,9,4
3,11,6


**DataFrame API:**

In [ ]:
per_month = (orders.filter(F.year("order_date") == 2024)
             .groupBy(F.month("order_date").alias("order_month"))
             .agg(F.countDistinct("order_id").alias("num_orders")))
avg_n = per_month.agg(F.avg("num_orders")).first()[0]
res = per_month.filter(F.col("num_orders") > avg_n).orderBy("order_month")
res.show()

+-----------+----------+
|order_month|num_orders|
+-----------+----------+
|          1|         4|
|          3|         5|
|          9|         4|
|         11|         6|
+-----------+----------+

**Why / traps:** Filter dates with a **range** (`>= start AND < end`) instead of `EXTRACT(YEAR ...) = 2024`. On partitioned tables the range lets the engine skip data (partition pruning).